# 05. SARIMAX Intervention and Counterfactual Analysis 

This notebook reads raw intervention, residual-diagnostic, and counterfactual outputs from Notebook 03. It creates report-ready SARIMAX intervention and counterfactual tables.

This notebook evaluates COVID-period and post-COVID intervention evidence. 
The analysis is intervention-based structural-change analysis: the break dates are defined by the research design rather than estimated as unknown breakpoints.

## A. Import and Path

In [7]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = Path('modeling_outputs')
FIGURES_DIR = Path('figures')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 160)

COUNTRY_DISPLAY = {
    'Australia': 'Australia',
    'Canada': 'Canada',
    'Ireland': 'Ireland',
    'New_Zealand': 'New Zealand',
    'New Zealand': 'New Zealand',
    'UK': 'United Kingdom',
    'US': 'United States',
}

MODEL_DISPLAY = {
    'Naive': 'Naive',
    'Seasonal_Naive': 'Seasonal Naive',
    'Seasonal Naive': 'Seasonal Naive',
    'ARIMA': 'ARIMA',
    'SARIMA': 'SARIMA',
    'SARIMAX': 'SARIMAX',
}

HORIZON_DISPLAY = {
    'h01_03': 'h01-03',
    'h04_12': 'h04-12',
    'h13_24': 'h13-24',
    'h25_plus': 'h25+',
}

def read_result(name: str) -> pd.DataFrame:
    candidates = [RESULTS_DIR / name, Path(name)]
    for path in candidates:
        if path.exists():
            return pd.read_csv(path)
    searched = ' or '.join(str(p) for p in candidates)
    raise FileNotFoundError(f'Cannot find {name}. Searched: {searched}. Run Notebook 03 first, then run this notebook.')

def require_columns(df: pd.DataFrame, cols, table_name: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f'{table_name} is missing required columns: {missing}. Available columns: {list(df.columns)}')

def add_display_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'country' in out.columns:
        out['country_label'] = out['country'].map(COUNTRY_DISPLAY).fillna(out['country'])
    if 'model_type' in out.columns:
        out['model_label'] = out['model_type'].map(MODEL_DISPLAY).fillna(out['model_type'])
    if 'best_test_model' in out.columns:
        out['best_test_model_label'] = out['best_test_model'].map(MODEL_DISPLAY).fillna(out['best_test_model'])
    if 'horizon_group' in out.columns:
        out['horizon_label'] = out['horizon_group'].map(HORIZON_DISPLAY).fillna(out['horizon_group'])
    return out


## B. Load raw SARIMAX and counterfactual outputs from Notebook 03

In [8]:

sarimax = read_result('03_final_anxiety_sarimax_intervention_table_enhanced.csv')
cf_monthly = read_result('03_anxiety_counterfactual_monthly_enhanced.csv')
cf_gap_raw = read_result('03_anxiety_counterfactual_covid_gap_summary_enhanced.csv')

# Residual diagnostics can be stored either in the full-sample SARIMAX table or in the residual diagnostics table.
try:
    diag_raw = read_result('anxiety_residual_diagnostics_enhanced.csv')
except FileNotFoundError:
    diag_raw = pd.DataFrame()

require_columns(sarimax, ['country'], 'final_anxiety_sarimax_intervention_table_enhanced.csv')
require_columns(cf_monthly, ['country', 'phase', 'counterfactual_version', 'observed', 'counterfactual_mean', 'gap', 'observed_above_counterfactual', 'observed_above_upper_95', 'counterfactual_exceeds_100'], 'anxiety_counterfactual_monthly_enhanced.csv')

sarimax = sarimax.replace([np.inf, -np.inf], np.nan)
cf_monthly = cf_monthly.replace([np.inf, -np.inf], np.nan)
cf_gap_raw = cf_gap_raw.replace([np.inf, -np.inf], np.nan)

display(sarimax.head())
display(cf_gap_raw.head())


,country,order,seasonal_order,exog_set,AIC,BIC,converged,resid_n,resid_mean,resid_std,LjungBox_p12,LjungBox_p24,ARCH_LM_p12,JarqueBera_p,status,early_covid_shock_coef,early_covid_shock_pvalue,covid_period_coef,covid_period_pvalue,covid_trend_coef,covid_trend_pvalue,post_covid_period_coef,post_covid_period_pvalue,post_covid_trend_coef,post_covid_trend_pvalue
0,Australia,"(1, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,1307.593828,1342.274429,True,264,0.102630,4.183652,0.062631,1.005659e-01,6.270711e-13,1.807964e-160,OK,-1.089193,0.630658,-2.060245,0.594017,-0.519486,0.029926,-23.760273,0.024893,-0.961909,0.005257
1,Canada,"(1, 1, 1)","(0, 0, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,1408.280279,1443.454808,True,264,0.053696,4.376930,0.019799,2.976631e-09,1.323972e-02,0.000000e+00,OK,-7.876626,0.075014,1.841985,0.606859,-0.318973,0.118511,-11.146011,0.267009,-0.646557,0.001795
2,Ireland,"(2, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,1376.663517,1414.812178,True,264,0.116920,4.694010,0.980827,8.510330e-01,1.176280e-05,1.408227e-82,OK,-4.680620,0.339403,-0.343463,0.962860,-0.615027,0.111888,-28.517145,0.101050,-0.844581,0.087418
3,New_Zealand,"(0, 1, 1)","(1, 0, 0, 12)",early_covid_shock+covid_period+covid_trend+pos...,1559.427179,1591.156255,True,264,0.071871,5.678170,0.038927,1.466380e-01,5.837526e-01,1.564583e-11,OK,0.435117,0.920681,-2.668641,0.719050,0.253800,0.662084,5.356142,0.835621,-0.893694,0.181171
4,UK,"(2, 1, 1)","(0, 0, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,1366.616931,1405.308913,True,264,0.037586,3.698107,0.200473,4.389158e-07,4.101939e-05,3.144295e-23,OK,-9.868752,0.000002,3.425563,0.131201,-0.314695,0.196395,-8.929540,0.395273,-0.526328,0.035090


,country,counterfactual_version,phase,n_months,mean_observed,mean_counterfactual,mean_gap,cumulative_gap,percent_gap_vs_counterfactual_mean,share_months_above_counterfactual,share_months_above_upper_95,share_counterfactual_exceeds_100
0,Australia,capped,COVID_period,41,86.707317,95.106490,-8.399173,-344.366073,-8.831335,0.024390,0.00000,0.000000
1,Australia,capped,Post_COVID,31,84.064516,99.781984,-15.717468,-487.241515,-15.751810,0.000000,0.00000,0.000000
2,Australia,logit,COVID_period,41,86.707317,91.805232,-5.097915,-209.014526,-5.552968,0.170732,0.04878,0.000000
3,Australia,logit,Post_COVID,31,84.064516,95.861464,-11.796948,-365.705393,-12.306247,0.000000,0.00000,0.000000
4,Australia,raw,COVID_period,41,86.707317,96.204991,-9.497674,-389.404644,-9.872330,0.024390,0.00000,0.317073


## C. SARIMAX intervention terms

The terms are `early_covid_shock`, `covid_period`, `covid_trend`, `post_covid_period`, and `post_covid_trend`. A significant positive term would support increased anxiety search behavior; a significant negative term supports decline or normalization.

In [9]:

coef_terms = ['early_covid_shock', 'covid_period', 'covid_trend', 'post_covid_period', 'post_covid_trend']
rows = []
for _, row in sarimax.iterrows():
    for term in coef_terms:
        coef = row.get(f'{term}_coef', np.nan)
        pval = row.get(f'{term}_pvalue', np.nan)
        rows.append({
            'country': row['country'],
            'term': term,
            'coef': coef,
            'pvalue': pval,
            'significant_5pct': bool(pd.notna(pval) and pval < 0.05),
            'direction': 'positive' if pd.notna(coef) and coef > 0 else 'negative' if pd.notna(coef) and coef < 0 else 'zero_or_missing',
        })
terms = add_display_columns(pd.DataFrame(rows))
sig = terms.loc[terms['significant_5pct']].copy().sort_values(['country', 'term']).reset_index(drop=True)
terms.to_csv(RESULTS_DIR / '05_sarimax_intervention_terms_long.csv', index=False)
sig.to_csv(RESULTS_DIR / '05_significant_sarimax_intervention_terms.csv', index=False)

display(terms)
display(sig)


,country,term,coef,pvalue,significant_5pct,direction,country_label
0,Australia,early_covid_shock,-1.089193,0.630658,False,negative,Australia
1,Australia,covid_period,-2.060245,0.594017,False,negative,Australia
2,Australia,covid_trend,-0.519486,0.029926,True,negative,Australia
3,Australia,post_covid_period,-23.760273,0.024893,True,negative,Australia
4,Australia,post_covid_trend,-0.961909,0.005257,True,negative,Australia
5,Canada,early_covid_shock,-7.876626,0.075014,False,negative,Canada
6,Canada,covid_period,1.841985,0.606859,False,positive,Canada
7,Canada,covid_trend,-0.318973,0.118511,False,negative,Canada
8,Canada,post_covid_period,-11.146011,0.267009,False,negative,Canada
9,Canada,post_covid_trend,-0.646557,0.001795,True,negative,Canada


,country,term,coef,pvalue,significant_5pct,direction,country_label
0,Australia,covid_trend,-0.519486,0.029926,True,negative,Australia
1,Australia,post_covid_period,-23.760273,0.024893,True,negative,Australia
2,Australia,post_covid_trend,-0.961909,0.005257,True,negative,Australia
3,Canada,post_covid_trend,-0.646557,0.001795,True,negative,Canada
4,UK,early_covid_shock,-9.868752,0.000002,True,negative,United Kingdom
5,UK,post_covid_trend,-0.526328,0.035090,True,negative,United Kingdom
6,US,post_covid_trend,-0.496986,0.049463,True,negative,United States


## D. Full-sample SARIMAX residual diagnostics

Diagnostics are extracted from the full-sample SARIMAX table when available. If not, the notebook falls back to the raw residual diagnostics file from Notebook 03. Missing diagnostic values are not treated as passes.

In [10]:

preferred_diag_cols = ['country', 'order', 'seasonal_order', 'exog_set', 'resid_n', 'resid_mean', 'resid_std', 'LjungBox_p12', 'LjungBox_p24', 'ARCH_LM_p12', 'JarqueBera_p']
if all(col in sarimax.columns for col in ['LjungBox_p24', 'ARCH_LM_p12', 'JarqueBera_p']):
    full_diag = sarimax[[c for c in preferred_diag_cols if c in sarimax.columns]].copy()
else:
    if diag_raw.empty:
        raise ValueError('No usable residual diagnostics found in SARIMAX table or anxiety_residual_diagnostics_enhanced.csv.')
    full_diag = diag_raw.copy()
    if 'model_type' in full_diag.columns:
        full_diag = full_diag.loc[full_diag['model_type'].eq('SARIMAX')].copy()

for col in preferred_diag_cols:
    if col not in full_diag.columns:
        full_diag[col] = np.nan
full_diag = full_diag[preferred_diag_cols].copy()
full_diag.insert(1, 'model_scope', 'full_sample_sarimax_intervention')
full_diag.insert(2, 'model_type', 'SARIMAX')
full_diag['lb24_pass_5pct'] = full_diag['LjungBox_p24'].notna() & (full_diag['LjungBox_p24'] > 0.05)
full_diag['arch_pass_5pct'] = full_diag['ARCH_LM_p12'].notna() & (full_diag['ARCH_LM_p12'] > 0.05)
full_diag['jb_pass_5pct'] = full_diag['JarqueBera_p'].notna() & (full_diag['JarqueBera_p'] > 0.05)
full_diag = add_display_columns(full_diag)
full_diag.to_csv(RESULTS_DIR / '05_full_sample_sarimax_residual_diagnostics.csv', index=False)
display(full_diag)


,country,model_scope,model_type,order,seasonal_order,exog_set,resid_n,resid_mean,resid_std,LjungBox_p12,LjungBox_p24,ARCH_LM_p12,JarqueBera_p,lb24_pass_5pct,arch_pass_5pct,jb_pass_5pct,country_label,model_label
0,Australia,full_sample_sarimax_intervention,SARIMAX,"(1, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,264,0.102630,4.183652,0.062631,1.005659e-01,6.270711e-13,1.807964e-160,True,False,False,Australia,SARIMAX
1,Canada,full_sample_sarimax_intervention,SARIMAX,"(1, 1, 1)","(0, 0, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,264,0.053696,4.376930,0.019799,2.976631e-09,1.323972e-02,0.000000e+00,False,False,False,Canada,SARIMAX
2,Ireland,full_sample_sarimax_intervention,SARIMAX,"(2, 1, 1)","(0, 1, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,264,0.116920,4.694010,0.980827,8.510330e-01,1.176280e-05,1.408227e-82,True,False,False,Ireland,SARIMAX
3,New_Zealand,full_sample_sarimax_intervention,SARIMAX,"(0, 1, 1)","(1, 0, 0, 12)",early_covid_shock+covid_period+covid_trend+pos...,264,0.071871,5.678170,0.038927,1.466380e-01,5.837526e-01,1.564583e-11,True,True,False,New Zealand,SARIMAX
4,UK,full_sample_sarimax_intervention,SARIMAX,"(2, 1, 1)","(0, 0, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,264,0.037586,3.698107,0.200473,4.389158e-07,4.101939e-05,3.144295e-23,False,False,False,United Kingdom,SARIMAX
5,US,full_sample_sarimax_intervention,SARIMAX,"(1, 1, 1)","(1, 1, 1, 12)",early_covid_shock+covid_period+covid_trend+pos...,264,0.083259,3.440927,0.063361,1.434593e-01,3.667517e-20,0.000000e+00,True,False,False,United States,SARIMAX


## E. Country-level SARIMAX intervention interpretation

In [11]:

interpret_rows = []
for country in sorted(sarimax['country'].dropna().unique()):
    sub = sig.loc[sig['country'].eq(country)]
    if sub.empty:
        txt = 'No statistically significant SARIMAX intervention term at the 5% level.'
        label = 'no significant intervention evidence'
    else:
        pieces = [f"{r.term} is {r.direction} (coef={r.coef:.3f}, p={r.pvalue:.3f})" for r in sub.itertuples()]
        txt = '; '.join(pieces) + '.'
        if (sub['direction'] == 'negative').all():
            label = 'negative or declining intervention evidence'
        elif (sub['direction'] == 'positive').all():
            label = 'positive intervention evidence'
        else:
            label = 'mixed intervention evidence'
    interpret_rows.append({
        'country': country,
        'country_label': COUNTRY_DISPLAY.get(country, country),
        'intervention_label': label,
        'sarimax_intervention_interpretation': txt,
    })
interp = pd.DataFrame(interpret_rows)
interp.to_csv(RESULTS_DIR / '05_country_intervention_interpretation.csv', index=False)
display(interp)


,country,country_label,intervention_label,sarimax_intervention_interpretation
0,Australia,Australia,negative or declining intervention evidence,"covid_trend is negative (coef=-0.519, p=0.030)..."
1,Canada,Canada,negative or declining intervention evidence,"post_covid_trend is negative (coef=-0.647, p=0..."
2,Ireland,Ireland,no significant intervention evidence,No statistically significant SARIMAX intervent...
3,New_Zealand,New Zealand,no significant intervention evidence,No statistically significant SARIMAX intervent...
4,UK,United Kingdom,negative or declining intervention evidence,"early_covid_shock is negative (coef=-9.869, p=..."
5,US,United States,negative or declining intervention evidence,"post_covid_trend is negative (coef=-0.497, p=0..."


## F. Counterfactual version summary

Raw SARIMA counterfactuals can exceed the Google Trends 0-100 scale. Capped and logit versions are therefore retained as bounded-scale robustness checks. The logit version is preferred for interpretation.

In [12]:

def summarize_counterfactual(g: pd.DataFrame) -> dict:
    mean_cf = g['counterfactual_mean'].mean()
    return {
        'n_months': int(len(g)),
        'mean_observed': float(g['observed'].mean()),
        'mean_counterfactual': float(mean_cf),
        'mean_gap': float(g['gap'].mean()),
        'cumulative_gap': float(g['gap'].sum()),
        'percent_gap_vs_counterfactual_mean': float(g['gap'].mean() / mean_cf * 100) if pd.notna(mean_cf) and mean_cf != 0 else np.nan,
        'share_months_above_counterfactual': float(g['observed_above_counterfactual'].mean()),
        'share_months_above_upper_95': float(g['observed_above_upper_95'].mean()),
        'share_counterfactual_exceeds_100': float(g['counterfactual_exceeds_100'].mean()),
    }

# Version-level summary across all countries, by COVID/Post-COVID phase.
summary_rows = []
for (version, phase), g in cf_monthly.groupby(['counterfactual_version', 'phase'], dropna=False):
    row = {'counterfactual_version': version, 'phase': phase}
    row.update(summarize_counterfactual(g))
    summary_rows.append(row)

# Full 2020-2025 summary across all countries.
for version, g in cf_monthly.groupby('counterfactual_version', dropna=False):
    row = {'counterfactual_version': version, 'phase': 'Full_2020_2025'}
    row.update(summarize_counterfactual(g))
    summary_rows.append(row)

cf_summary = pd.DataFrame(summary_rows).sort_values(['counterfactual_version', 'phase']).reset_index(drop=True)
cf_summary.to_csv(RESULTS_DIR / '05_counterfactual_version_summary.csv', index=False)
display(cf_summary)


,counterfactual_version,phase,n_months,mean_observed,mean_counterfactual,mean_gap,cumulative_gap,percent_gap_vs_counterfactual_mean,share_months_above_counterfactual,share_months_above_upper_95,share_counterfactual_exceeds_100
0,capped,COVID_period,246,86.817073,92.301205,-5.484131,-1349.096330,-5.941560,0.215447,0.008130,0.000000
1,capped,Full_2020_2025,432,85.800926,95.454030,-9.653104,-4170.141067,-10.112831,0.122685,0.004630,0.000000
2,capped,Post_COVID,186,84.456989,99.623896,-15.166907,-2821.044737,-15.224166,0.000000,0.000000,0.000000
3,logit,COVID_period,246,86.817073,89.664846,-2.847773,-700.552154,-3.176019,0.386179,0.089431,0.000000
4,logit,Full_2020_2025,432,85.800926,91.863826,-6.062900,-2619.172796,-6.599878,0.252315,0.062500,0.000000
5,logit,Post_COVID,186,84.456989,94.772154,-10.315165,-1918.620642,-10.884173,0.075269,0.026882,0.000000
6,raw,COVID_period,246,86.817073,93.069492,-6.252419,-1538.095136,-6.718012,0.215447,0.008130,0.186992
7,raw,Full_2020_2025,432,85.800926,100.378797,-14.577871,-6297.640303,-14.522859,0.122685,0.004630,0.490741
8,raw,Post_COVID,186,84.456989,110.045942,-25.588953,-4759.545166,-23.252972,0.000000,0.000000,0.892473


## G. Country-phase counterfactual summary

In [13]:

country_phase_rows = []
for (country, version, phase), g in cf_monthly.groupby(['country', 'counterfactual_version', 'phase'], dropna=False):
    row = {'country': country, 'counterfactual_version': version, 'phase': phase}
    row.update(summarize_counterfactual(g))
    country_phase_rows.append(row)

for (country, version), g in cf_monthly.groupby(['country', 'counterfactual_version'], dropna=False):
    row = {'country': country, 'counterfactual_version': version, 'phase': 'Full_2020_2025'}
    row.update(summarize_counterfactual(g))
    country_phase_rows.append(row)

cf_by_country_phase = add_display_columns(pd.DataFrame(country_phase_rows))
cf_by_country_phase.to_csv(RESULTS_DIR / '05_counterfactual_by_country_phase.csv', index=False)

logit_phase = cf_by_country_phase.loc[cf_by_country_phase['counterfactual_version'].eq('logit')].copy()
logit_phase.to_csv(RESULTS_DIR / '05_logit_counterfactual_by_country_phase.csv', index=False)

display(logit_phase.sort_values(['country', 'phase']))


,country,counterfactual_version,phase,n_months,mean_observed,mean_counterfactual,mean_gap,cumulative_gap,percent_gap_vs_counterfactual_mean,share_months_above_counterfactual,share_months_above_upper_95,share_counterfactual_exceeds_100,country_label
2,Australia,logit,COVID_period,41,86.707317,91.805232,-5.097915,-209.014526,-5.552968,0.170732,0.048780,0.0,Australia
37,Australia,logit,Full_2020_2025,72,85.569444,93.551666,-7.982221,-574.719919,-8.532420,0.097222,0.027778,0.0,Australia
3,Australia,logit,Post_COVID,31,84.064516,95.861464,-11.796948,-365.705393,-12.306247,0.000000,0.000000,0.0,Australia
8,Canada,logit,COVID_period,41,88.829268,88.516740,0.312528,12.813666,0.353073,0.560976,0.097561,0.0,Canada
40,Canada,logit,Full_2020_2025,72,87.027778,89.976587,-2.948809,-212.314283,-3.277308,0.347222,0.055556,0.0,Canada
9,Canada,logit,Post_COVID,31,84.645161,91.907353,-7.262192,-225.127949,-7.901644,0.064516,0.000000,0.0,Canada
14,Ireland,logit,COVID_period,41,82.365854,91.999818,-9.633964,-394.992525,-10.471721,0.024390,0.000000,0.0,Ireland
43,Ireland,logit,Full_2020_2025,72,80.041667,94.516033,-14.474367,-1042.154403,-15.314192,0.013889,0.000000,0.0,Ireland
15,Ireland,logit,Post_COVID,31,76.967742,97.843932,-20.876190,-647.161878,-21.336213,0.000000,0.000000,0.0,Ireland
20,New_Zealand,logit,COVID_period,41,89.073171,92.837522,-3.764351,-154.338385,-4.054773,0.292683,0.048780,0.0,New Zealand


## H. Intervention and counterfactual summary for report

In [14]:

n_sig_terms = int(len(sig))
n_sig_countries = int(sig['country'].nunique()) if not sig.empty else 0
neg = int((sig['direction'] == 'negative').sum()) if not sig.empty else 0
pos = int((sig['direction'] == 'positive').sum()) if not sig.empty else 0
pass_count = int(full_diag['lb24_pass_5pct'].sum()) if 'lb24_pass_5pct' in full_diag.columns else 0
raw_full = cf_summary.loc[(cf_summary['counterfactual_version'].eq('raw')) & (cf_summary['phase'].eq('Full_2020_2025'))]
raw_exceed = float(raw_full['share_counterfactual_exceeds_100'].iloc[0]) if not raw_full.empty else np.nan
logit_full = logit_phase.loc[logit_phase['phase'].eq('Full_2020_2025')]
negative_logit_countries = int((logit_full['mean_gap'] < 0).sum()) if not logit_full.empty else 0

summary = pd.DataFrame({'sarimax_intervention_counterfactual_summary': [
    f'SARIMAX intervention results identify {n_sig_terms} statistically significant terms at the 5% level across {n_sig_countries}/6 countries.',
    f'Significant terms are negative or declining in the main results ({neg} negative; {pos} positive), so the intervention evidence does not support a uniform positive COVID-period increase in anxiety searches.',
    f'Full-sample SARIMAX residual diagnostics pass the Ljung-Box p24 test in {pass_count}/6 countries; inference should be cautious where residual autocorrelation remains.',
    f'Raw counterfactual forecasts exceed the Google Trends upper bound in {raw_exceed:.1%} of full-period observations; capped and logit counterfactuals are preferred for interpretation.' if pd.notna(raw_exceed) else 'Raw counterfactual upper-bound exceedance could not be computed.',
    f'The logit counterfactual shows negative full-period mean gaps in {negative_logit_countries}/6 countries.' if not logit_full.empty else 'The logit counterfactual country summary could not be computed.'
]})
summary.to_csv(RESULTS_DIR / '05_sarimax_intervention_counterfactual_summary.csv', index=False)
# Backward-compatible filename used by the previous report pipeline.
summary.rename(columns={'sarimax_intervention_counterfactual_summary': 'structural_break_sarimax_summary'}).to_csv(RESULTS_DIR / '05_structural_break_sarimax_summary.csv', index=False)
display(summary)


,sarimax_intervention_counterfactual_summary
0,SARIMAX intervention results identify 7 statis...
1,Significant terms are negative or declining in...
2,Full-sample SARIMAX residual diagnostics pass ...
3,Raw counterfactual forecasts exceed the Google...
4,The logit counterfactual shows negative full-p...
